# 🤗 Real-World NLP with Hugging Face
### Undergraduate Workshop — 60 Minutes

---

**What you'll build today:**

| # | Application | Real-World Use Case | Time |
|---|-------------|---------------------|------|
| 1 | Sentiment Analysis | Amazon/Flipkart product reviews | ~10 min |
| 2 | Named Entity Recognition | Resume parsing & news extraction | ~10 min |
| 3 | Question Answering | Customer support bots | ~10 min |
| 4 | Text Summarization | News digest / Research abstracts | ~10 min |
| 5 | Zero-Shot Classification | Content moderation, tagging | ~10 min |
| 6 | Mini Project | Build a Review Analyzer | ~10 min |

> **No GPU needed.** All models run on CPU for this workshop.

---


## 🛠️ Setup — Run This First!

Install the Hugging Face `transformers` library and a couple of helpers. This takes ~1 minute.

In [1]:
# Core imports
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print("✅ Hugging Face Transformers loaded!")
print("\n🔑 Key concept: The pipeline() function is your main tool.")
print("   It wraps a pre-trained model into one easy-to-use function.")

✅ Hugging Face Transformers loaded!

🔑 Key concept: The pipeline() function is your main tool.
   It wraps a pre-trained model into one easy-to-use function.


---
## 💡 Core Concept: What is a Pipeline?

```
Your Text  →  [Tokenizer]  →  [Model]  →  [Post-processing]  →  Result
```

Hugging Face **pipelines** bundle all 3 steps so you can go from raw text to a prediction in just **2 lines of code**. Under the hood, these use **transformer models** pre-trained on billions of words.

---

## 📦 Section 1 — Sentiment Analysis
### Real-World Use: E-commerce Review Systems

**Scenario:** You work at a startup. 10,000 product reviews come in daily. You need to flag negative reviews instantly for customer support to act on.

> **Model used:** `distilbert-base-uncased-finetuned-sst-2-english` (fine-tuned on movie reviews, generalizes well)

In [ ]:
# Load the sentiment pipeline — first run downloads the model (~250MB)
print("Loading sentiment model... (first time may take ~30 seconds)")
sentiment = pipeline("sentiment-analysis")
print("✅ Model ready!")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading sentiment model... (first time may take ~30 seconds)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


✅ Model ready!


: 

In [ ]:
# Basic usage
reviews = [
    "This phone has an amazing camera and battery life!",
    "Broke after 2 days. Worst purchase I ever made.",
    "Decent product, nothing extraordinary but works fine."
]

results = sentiment(reviews)

print("=" * 55)
print(f"{'Review':<40} {'Sentiment':<10} {'Confidence'}")
print("=" * 55)
for review, result in zip(reviews, results):
    label = result['label']
    score = result['score']
    emoji = "😊" if label == "POSITIVE" else "😠"
    print(f"{review[:38]:<40} {emoji} {label:<8} {score:.1%}")

In [ ]:
# Real-world application: Auto-flagging negative reviews for support team
def triage_reviews(reviews, threshold=0.90):
    """Automatically categorize reviews and flag urgent negatives."""
    results = sentiment(reviews)
    urgent_flags = []
    
    print("📋 REVIEW TRIAGE REPORT")
    print("-" * 50)
    
    for i, (review, result) in enumerate(zip(reviews, results)):
        label = result['label']
        score = result['score']
        
        if label == 'NEGATIVE' and score > threshold:
            urgent_flags.append(i)
            tag = "🚨 URGENT"
        elif label == 'NEGATIVE':
            tag = "⚠️  NEGATIVE"
        else:
            tag = "✅ POSITIVE"
        
        print(f"[{i+1}] {tag} ({score:.0%}) — {review[:45]}...")
    
    print(f"\n📊 Summary: {len(urgent_flags)} urgent reviews need immediate attention.")
    return urgent_flags

sample_reviews = [
    "Absolutely love this laptop! Best investment ever.",
    "The charger stopped working after a week. Very disappointed.",
    "Good value for money, fast delivery.",
    "Scam! Product is nothing like the description. Demanding refund!",
    "Average. Not great, not terrible."
]

flags = triage_reviews(sample_reviews)

### ✏️ Try It Yourself!
Add 2-3 of your own reviews to `sample_reviews` above and re-run the cell. Try writing something ambiguous — can the model handle sarcasm?

---
## 🏷️ Section 2 — Named Entity Recognition (NER)
### Real-World Use: Resume Parsing & News Monitoring

**Scenario 1:** A recruitment platform needs to auto-extract skills, companies, and locations from thousands of resumes.  
**Scenario 2:** A newsroom monitors articles for mentions of specific people and organizations.

> **NER** identifies and classifies *named entities* like **persons (PER)**, **organizations (ORG)**, **locations (LOC)**, and **miscellaneous (MISC)**.

In [ ]:
print("Loading NER model...")
ner = pipeline("ner", aggregation_strategy="simple")
print("✅ NER model ready!")

In [ ]:
# Application 1: News Article Entity Extraction
news_article = """
Sundar Pichai, CEO of Google, announced a partnership with Samsung Electronics 
at a tech summit held in San Francisco last Tuesday. The deal involves integrating 
Google's Gemini AI into Samsung devices across India and South Korea by 2025.
"""

entities = ner(news_article)

print("📰 NEWS ARTICLE ENTITY EXTRACTION")
print("-" * 45)

# Group entities by type
entity_groups = {}
for ent in entities:
    etype = ent['entity_group']
    word = ent['word']
    if etype not in entity_groups:
        entity_groups[etype] = []
    if word not in entity_groups[etype]:  # deduplicate
        entity_groups[etype].append(word)

icons = {'PER': '👤 People', 'ORG': '🏢 Organizations', 
         'LOC': '📍 Locations', 'MISC': '🔖 Misc'}

for etype, words in entity_groups.items():
    label = icons.get(etype, etype)
    print(f"  {label}: {', '.join(words)}")

In [ ]:
# Application 2: Resume / LinkedIn Profile Parsing
resume_snippet = """
Priya Sharma worked as a Data Scientist at Infosys in Bengaluru for 3 years, 
then moved to Microsoft in Hyderabad. She holds a degree from IIT Bombay 
and completed a certification from Coursera.
"""

entities = ner(resume_snippet)

print("📄 RESUME PARSING OUTPUT")
print("-" * 45)

entity_groups = {}
for ent in entities:
    etype = ent['entity_group']
    word = ent['word']
    if etype not in entity_groups:
        entity_groups[etype] = []
    if word not in entity_groups[etype]:
        entity_groups[etype].append(word)

for etype, words in entity_groups.items():
    label = icons.get(etype, etype)
    print(f"  {label}: {', '.join(words)}")

print("\n💡 This is how LinkedIn, Naukri.com, and ATS systems auto-parse your CV!")

---
## ❓ Section 3 — Question Answering
### Real-World Use: Customer Support Bots & Document Search

**Scenario:** A bank deploys a chatbot. Customers ask questions. The bot finds the answer *within* a policy document — without needing to memorize it.

> This is **extractive QA**: the model finds the answer span *directly from a given context passage*. This is how early versions of Google's Featured Snippets and enterprise chatbots work.

In [ ]:
print("Loading Question Answering model...")
qa = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)
print("✅ QA model ready!")

In [ ]:
# Simulated bank policy document
bank_policy = """
Our savings account offers an interest rate of 6.5% per annum, compounded quarterly.
The minimum balance required to open an account is Rs. 1000. Customers can withdraw 
up to Rs. 50,000 per day from ATMs. International transactions attract a fee of 3.5% 
of the transaction amount. Fixed deposits are available for terms ranging from 7 days 
to 10 years, with rates between 4% and 7.8%. Account holders must provide a valid 
Aadhaar card and PAN card to complete KYC verification.
"""

# Customer questions
customer_questions = [
    "What is the interest rate on savings account?",
    "How much can I withdraw from an ATM per day?",
    "What documents do I need to open an account?",
    "What is the minimum balance to open an account?"
]

print("🏦 BANK CUSTOMER SUPPORT BOT")
print("=" * 55)

for question in customer_questions:
    result = qa(question=question, context=bank_policy)
    confidence = result['score']
    answer = result['answer']
    
    conf_bar = "█" * int(confidence * 10) + "░" * (10 - int(confidence * 10))
    
    print(f"\n❓ {question}")
    print(f"💬 {answer}")
    print(f"   Confidence: [{conf_bar}] {confidence:.0%}")

In [ ]:
# 🧪 Try your own question!
my_question = "What is the fee for international transactions?"

result = qa(question=my_question, context=bank_policy)
print(f"❓ {my_question}")
print(f"💬 Answer: {result['answer']}")
print(f"   Confidence: {result['score']:.1%}")

# TODO: Change my_question to something else and re-run!

---
## 📝 Section 4 — Text Summarization
### Real-World Use: News Digest Apps & Research Tools

**Scenario:** An app like Inshorts or Google News needs to condense long articles into 2-3 sentences. A researcher wants a quick abstract of a paper.

> **Abstractive summarization** generates *new sentences* that capture the main idea — the model doesn't just copy sentences from the original.

In [ ]:
# distilbart-cnn-12-6: lightweight, fast, works great on CPU
print("Loading summarization model... (first run ~600MB, may take ~1 min)")
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6"
)
print("✅ Summarization model ready!")

In [ ]:
# A real-ish news article
article = """
India's renewable energy sector saw record investments in 2024, with solar power 
installations crossing 80 gigawatts for the first time. The government's push under 
the National Solar Mission has attracted over $12 billion in foreign direct investment, 
with companies like Adani Green Energy and Tata Power leading the expansion. 
Several new solar parks in Rajasthan and Gujarat are expected to add another 
20 GW by 2026. Experts note that India is on track to meet its COP26 commitment 
of achieving 500 GW of non-fossil fuel capacity by 2030, though challenges remain 
in grid infrastructure and land acquisition. The growth of solar has also created 
over 300,000 new jobs in manufacturing and installation across rural regions, 
contributing significantly to local economies.
"""

print(f"📄 ORIGINAL ({len(article.split())} words)")
print("-" * 50)
print(article.strip())

summary = summarizer(article, max_length=80, min_length=30, do_sample=False)
summary_text = summary[0]['summary_text']

print(f"\n⚡ SUMMARY ({len(summary_text.split())} words — {len(summary_text.split())/len(article.split()):.0%} of original)")
print("-" * 50)
print(summary_text)

In [ ]:
# Controlling summary length — useful for different output formats
print("📱 INSHORTS FORMAT (very short):")
short = summarizer(article, max_length=40, min_length=20, do_sample=False)
print(short[0]['summary_text'])

print("\n📰 NEWSPAPER BRIEF (medium):")
medium = summarizer(article, max_length=80, min_length=50, do_sample=False)
print(medium[0]['summary_text'])

print("\n📚 RESEARCH ABSTRACT (longer):")
long = summarizer(article, max_length=120, min_length=70, do_sample=False)
print(long[0]['summary_text'])

---
## 🎯 Section 5 — Zero-Shot Classification
### Real-World Use: Content Moderation & Auto-Tagging

**Scenario:** A social media platform wants to auto-tag posts into topics (tech, sports, politics) — but new topics keep appearing. Retraining a model every time is expensive.

> **Zero-Shot** means the model classifies text into categories it was *never explicitly trained on*. You just describe the labels in plain English!

In [ ]:
print("Loading Zero-Shot Classification model...")
classifier = pipeline("zero-shot-classification", 
                      model="facebook/bart-large-mnli")
print("✅ Zero-shot classifier ready!")

In [ ]:
# Auto-tagging social media / news posts
posts = [
    "India wins the T20 World Cup after a thrilling final against Australia!",
    "The RBI has cut repo rates by 25 basis points to boost economic growth.",
    "Scientists discover a new exoplanet that may support liquid water.",
    "New law mandates electric vehicles for all government fleets by 2027.",
    "Shah Rukh Khan's latest film shatters box office records worldwide."
]

topic_labels = ["sports", "economy", "technology", "science", "politics", "entertainment"]

print("📱 AUTO-TAGGING SOCIAL MEDIA POSTS")
print("=" * 55)
print(f"Labels used: {', '.join(topic_labels)}\n")

for post in posts:
    result = classifier(post, candidate_labels=topic_labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    second_label = result['labels'][1]
    second_score = result['scores'][1]
    
    print(f"📌 {post[:55]}...")
    print(f"   → 🏷️  [{top_label.upper()}] {top_score:.0%}  |  2nd: [{second_label}] {second_score:.0%}")
    print()

In [ ]:
# Content Moderation — detecting harmful content types
content_samples = [
    "Join our community for free coding tutorials every Saturday!",
    "Buy cheap medicines without prescription, no questions asked.",
    "Flash sale! 90% off on all electronics. Limited time offer!",
    "How to make money fast without working — guaranteed results."
]

moderation_labels = ["safe content", "spam", "misinformation", "illegal activity", "advertisement"]

print("🚨 CONTENT MODERATION SYSTEM")
print("=" * 55)

for content in content_samples:
    result = classifier(content, candidate_labels=moderation_labels)
    top = result['labels'][0]
    score = result['scores'][0]
    
    flag = "✅" if top == "safe content" else "🚩"
    print(f"{flag} \"{content[:50]}...\"")
    print(f"   → Classified as: [{top.upper()}] — {score:.0%} confidence\n")

---
## 🚀 Section 6 — Mini Project: Smart Review Analyzer

**Put it all together!** You'll build a small pipeline that:
1. Analyzes the **sentiment** of a product review
2. Extracts **entities** (product names, brands, locations mentioned)
3. **Classifies** the review into a product category
4. Generates a structured **report**

This is a simplified version of what platforms like Amazon and Flipkart use for review intelligence.

In [ ]:
def analyze_review(review_text):
    """
    Full NLP pipeline for analyzing a product review.
    Combines sentiment, NER, and zero-shot classification.
    """
    print("\n" + "=" * 60)
    print("🔍 SMART REVIEW ANALYZER")
    print("=" * 60)
    print(f"📝 Review: {review_text}")
    print("-" * 60)
    
    # Step 1: Sentiment Analysis
    sent_result = sentiment(review_text)[0]
    sentiment_label = sent_result['label']
    sentiment_score = sent_result['score']
    sent_emoji = "😊" if sentiment_label == "POSITIVE" else "😠"
    
    print(f"\n{sent_emoji} SENTIMENT: {sentiment_label} ({sentiment_score:.1%} confidence)")
    
    # Step 2: Named Entity Recognition
    entities = ner(review_text)
    entity_groups = {}
    for ent in entities:
        etype = ent['entity_group']
        word = ent['word']
        if etype not in entity_groups:
            entity_groups[etype] = []
        if word not in entity_groups[etype]:
            entity_groups[etype].append(word)
    
    if entity_groups:
        print("\n🏷️  EXTRACTED ENTITIES:")
        for etype, words in entity_groups.items():
            label = icons.get(etype, etype)
            print(f"   {label}: {', '.join(words)}")
    else:
        print("\n🏷️  EXTRACTED ENTITIES: None detected")
    
    # Step 3: Product Category Classification
    categories = ["electronics", "clothing", "food & beverages", 
                  "home appliances", "books", "beauty & health"]
    cat_result = classifier(review_text, candidate_labels=categories)
    top_category = cat_result['labels'][0]
    cat_score = cat_result['scores'][0]
    
    print(f"\n📂 PRODUCT CATEGORY: {top_category.upper()} ({cat_score:.0%} confidence)")
    
    # Step 4: Action Recommendation
    print("\n📋 RECOMMENDED ACTION:")
    if sentiment_label == "NEGATIVE" and sentiment_score > 0.90:
        print("   🚨 HIGH PRIORITY — Escalate to customer support team immediately")
    elif sentiment_label == "NEGATIVE":
        print("   ⚠️  MODERATE — Queue for follow-up within 24 hours")
    elif sentiment_score > 0.95:
        print("   ⭐ FEATURE — Eligible for 'Verified Positive Review' badge")
    else:
        print("   ✅ ROUTINE — Archive and include in aggregate scoring")
    
    print("=" * 60)


# Test with different reviews
test_reviews = [
    "The Samsung Galaxy S24 is an absolute beast! Amazing display, camera is superb, and battery lasts all day. Bought it from Croma in Bangalore. 10/10!",
    "Zara jacket received completely damaged. The zipper broke on day one. Very poor quality for the price paid. Requesting full refund.",
]

for review in test_reviews:
    analyze_review(review)

In [ ]:
# 🎯 YOUR TURN — Write your own review and analyze it!
my_review = """Write your product review here and run this cell!"""

analyze_review(my_review)

---
## 🌟 Bonus — What's Under the Hood?

All models in this notebook are based on the **Transformer architecture** (Vaswani et al., 2017).

| Model Used | Architecture | Params | Pre-trained On |
|-----------|-------------|--------|----------------|
| Sentiment | DistilBERT | ~67M | BookCorpus + Wikipedia |
| NER | BERT | ~110M | CoNLL-2003 |
| QA | DistilBERT | ~67M | SQuAD v1.1 |
| Summarization | BART | ~400M | CNN/DailyMail |
| Zero-Shot | BART-MNLI | ~400M | MultiNLI |

### Key Insight: Fine-Tuning
```
Pre-train on HUGE corpus → Fine-tune on SMALL task-specific data → Deploy
   (weeks, expensive)       (hours, cheap)                        (seconds)
```
This is why you can run powerful NLP in a classroom — all the expensive training was done once and shared!

---
## 🎓 Workshop Wrap-Up

### What You Built Today

| Application | Business Problem Solved |
|-------------|------------------------|
| ✅ Sentiment Analysis | Auto-triage e-commerce reviews |
| ✅ NER | Resume parsing, news monitoring |
| ✅ Question Answering | Intelligent customer support bots |
| ✅ Summarization | News digest, research abstracts |
| ✅ Zero-Shot Classification | Content moderation, auto-tagging |
| ✅ Mini Project | End-to-end smart review analyzer |

### Next Steps

1. **Explore more models:** [huggingface.co/models](https://huggingface.co/models) — 500,000+ models!
2. **Fine-tune on your own data:** [Hugging Face Course](https://huggingface.co/learn/nlp-course) (free)
3. **Try Indian language NLP:** Look for models on Indic NLP Library or AI4Bharat
4. **Build an app:** Combine these pipelines with **Gradio** or **Streamlit** for a web UI in 10 lines

```python
# Deploy a sentiment analyzer as a web app in minutes!
# pip install gradio
import gradio as gr
demo = gr.Interface(fn=sentiment, inputs="text", outputs="label")
demo.launch()
```

---
*Built with 🤗 Hugging Face Transformers — Workshop for Undergraduate Students*